# មេរៀន ០២ - ការស្វែងយល់អំពី Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** គឺជាហ្រេះម៉ាវ្នភាសមួយសម្រាប់ការបង្កើតភ្នាក់ងារបញ្ញាសិប្បនិម្មិត។ វាប្រើរចនាសម្ព័ន្ធស្អាត ដែលអាចផ្សំបានដោយមានប្លុកកសាងមូលដ្ឋានបួនគឺ៖

- **ឧបករណ៍អតិថិជន (Client)** – ធ្វើការតភ្ជាប់ទៅកាន់ចំណុចប្រទាក់ម៉ូដែល AI ហើយដោះស្រាយការទំនាក់ទំនង
- **ភ្នាក់ងារ (Agent)** – រុំ client ជាមួយនឹងសេចក្ដីណែនាំ និងកំណត់ក្រុមហ៊ុនឧបករណ៍
- **ឧបករណ៍ (Tools)** – ពង្រីកសមត្ថភាពភ្នាក់ងារជាមួយមុខងារត្រឹមត្រូវដែលម៉ូដែលអាចហៅបាន
- **សកម្មភាព (Session)** – រក្សាទុកប្រវត្តិការសន្ទនាសម្រាប់អន្តរប្រតិបត្តិការជាច្រើនជុំ

នៅក្នុងមេរៀននេះ យើងនឹងបង្កើត **ភ្នាក់ងារការកក់ដំណើរកំសាន្ដ** ដែលបញ្ជាក់ពីកម្រិតភាពមាននៅគោលដៅ ដោយប្រើគំនិតទាំងនេះ។


## ការតំឡើង


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ការយល់ដឹងអំពីស្ថาปត្យកម្ម Agent Framework

Microsoft Agent Framework អនុវត្តតាមស្ថាបត្យកម្មមានស្រទាប់៖

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `FoundryChatClient` មួយភ្ជាប់ទៅកាន់ការចេញផ្សាយ Azure OpenAI។ វាចាំបាច់សម្រាប់ការផ្ទៀងផ្ទាត់ អត្ថបទសំណើ និងការវិភាគចម្លើយ។
2. **Agent** – ត្រូវបានបង្កើតពី client តាមរយៈ `provider.create_agent()`, agent បញ្ចូលការចូលប្រើម៉ូឌែលជាមួយនិងការណែនាំ (system prompt) និងឧបករណ៍។
3. **Tools** – មុខងារ Python ដែលត្រូវបានតស៊ូជាមួយ `@tool` ដែល agent អាចហៅដើម្បីអនុវត្តកម្មវិធី ឬទទួលបានទិន្នន័យ។
4. **Session** – អ объект `AgentSession` (ត្រូវបានបង្កើតតាមរយៈ `agent.create_session()`) ដែលរក្សាប្រវត្តិសន្ទនា អនុញ្ញាតឲ្យមានការពិភាក្សាច្រើនជំហាន ដែល agent ចាំបន្ទ_CONTEXTមុន។

មកផ្គុំគ្នារបស់រាល់ស្រទាប់ជាកម្រិត។


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ការបន្ថែមឧបករណ៍ជាមួយ @tool Decorator

ឧបករណ៍អនុញ្ញាតឲ្យភ្នាក់ងារប្រតិបត្តិការ លើសពីការបង្កើតអត្ថបទ។ `@tool` decorator បម្លែងមុខងារ Python ទៀងទាត់ទៅជារឿងដែលភ្នាក់ងារអាចហៅបាន។

ចំណុចសំខាន់៖
- ប្រើ `Annotated[type, "description"]` ដើម្បីឲ្យម៉ូដែលយល់ពីប៉ារ៉ាម៉ែត្រនីមួយៗ។
- docstring ក្លាយជាវិពណ៌នាឧបករណ៍ដែលម៉ូដែលមើលឃើញ។
- `approval_mode="never_require"` មានន័យថាឧបករណ៍ដំណើរការដោយស្វ័យប្រវត្តិដោយគ្មានការបញ្ជាក់ពីអ្នកប្រើ។


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ការបង្កើតភ្នាក់ងារជាមួយឧបករណ៍

ឥឡូវនេះយើងចែកចាយអតិថិជន, សារណែនាំ, និងឧបករណ៍ជារួមទៅក្នុងភ្នាក់ងារ។ "សារណែនាំ" ដំណើរការជាប្រភេទការជំរុញប្រព័ន្ធ — ពួកវាកំណត់អត្តសញ្ញាណនិងអាកប្បកិរិយារបស់ភ្នាក់ងារ។


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ជជែកពហុជើងជាមួយសម័យ

`AgentSession` មួយ (បង្កើតតាមរយៈ `agent.create_session()`) តាមដានសារទាំងអស់ក្នុងការជជែក។ ដោយផ្តល់សម័យដដែលទៅនឹងការហៅ `agent.run()` ហើយ អ្នកតំណាងអាចចូលដំណើរការប្រវត្តិស្វែងរកការជជែកទាំងមូល និងអាចយោងទៅកាន់សារមុនៗបាន។

យើងផ្តល់ `tools=[check_destination_availability]` ដើម្បីឲ្យអ្នកតំណាងអាចហៅកម្មវិធីពិនិត្យកម្រិតភាពមាននៅក្នុងរៀងត(turn)នីមួយៗបាន។


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## សេចក្តីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានស្វែងយល់ពីស្នោងចំនួនបួននៃ Microsoft Agent Framework៖

| គំនិត | អ្វីដែលអ្នកបានរៀន |
|---------|------------------|
| **អតិថិជន** | `FoundryChatClient` ចាប់ភ្ជាប់ទៅAzure OpenAI ជាមួយការផ្ទៀងផ្ទាត់ដោយគ្រីឌិនស្យាល់ |
| **ភ្នាក់ងារ** | `provider.create_agent()` ប្រមូលផ្តុំការតភ្ជាប់ម៉ូដែលជាមួយការណែនាំ និងឈ្មោះ |
| **ឧបករណ៍** | `@tool` decorator បង្ហាញមុខងារ Python សម្រាប់ភ្នាក់ងារ​ហៅឲ្យប្រើប្រាស់ |
| **សម័យ** | `agent.create_session()` ធ្វើការសម្រេចគ្នានៃប្រវត្តិសន្ទនា​​​គ្នា​នៅក្នុងជើងច្រើន |

លំនឹងបង្កើតទាំងនេះធ្វើការរួមបញ្ចូលគ្នាដើម្បីបង្កើតភ្នាក់ងារដែលអាចរក្សាសន្ទនាប្រកបដោយធម្មជាតិ ហៅមុខងារខាងក្រៅ និងរក្សាបរិបទ — ជាមូលដ្ឋានសម្រាប់លម្អិតលំអិតលើអ្នកភ្នាក់ងារថ្នាក់ខ្ពស់នៅមេរៀនក្រោយៗ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
